# Lesson 8.2 · Hyperparameters and diagnostics

A forest has few hyperparameters and almost all of them are robust: that is why it is an excellent "first attempt" model. This lesson says which ones matter, which do not, and how to diagnose the forest with the OOB error.

**What you will learn**

- `n_estimators`: it is a budget, not a hyperparameter to optimize.
- `max_features`, `min_samples_leaf`, `max_depth`: where the gains are.
- Importances: impurity (with its flaws), permutation, and how they behave with correlated features.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Number of trees: a budget

With `warm_start=True` you can add trees to an already trained forest and watch how the OOB error settles: it drops quickly in the first 50-100 trees and then stays flat. It is not "optimized": you choose a number high enough (100-500) and pay in time and memory.

In [ ]:
f = RandomForestClassifier(warm_start=True, oob_score=True, random_state=0, n_jobs=-1)
rows = []
for n in [5, 10, 20, 50, 100, 200, 400]:
    f.set_params(n_estimators=n).fit(X_tr, y_tr)
    rows.append({"trees": n, "OOB error": 1 - f.oob_score_, "test error": 1 - f.score(X_te, y_te)})
pd.DataFrame(rows).set_index("trees").round(4).T

## `max_features` and the minimum leaf size

- **`max_features`**: `"sqrt"` (default for classification), `1.0` = all the features (equivalent to **bagging**), or a fraction. It is the hyperparameter with the clearest effect on the correlation between trees: search it on a small grid.
- **`min_samples_leaf`**: larger leaves = smoother trees, less variance. With many noisy rows it can help; forests tolerate deep trees well, so the gain is usually modest.
- **`max_depth`**: it is rarely needed to limit it; look at `min_samples_leaf` first.

In [ ]:
for mf in [1, 2, "sqrt", 6, 12]:
    f = RandomForestClassifier(200, max_features=mf, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    print(f"max_features={str(mf):5s} | OOB error {1 - f.oob_score_:.4f} | test error {1 - f.score(X_te, y_te):.4f}")

## Feature importances

`feature_importances_` is the average of the impurity importances of the individual trees: it has the flaws seen in module 7 (it rewards features with many distinct values, and it is computed on the training set). The alternative is the **permutation importance** on data not used for training
(test set, or the OOB). With **correlated features** both split the importance among the copies: when one copy is shuffled the model can use the other, and the permutation **underestimates** each of them (exercise C2).

In [ ]:
f = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
perm = permutation_importance(f, X_te, y_te, n_repeats=10, random_state=0)
pd.DataFrame({"impurity": f.feature_importances_, "permutation (test)": perm.importances_mean}, index=[f"x{i}" for i in range(12)]).round(3).T

        > **Common mistake.** Trusting the defaults and a single experiment. In `RandomForestRegressor` `max_features` is 1.0 (all the features): without changing it, it is bagging, not a random forest (exercise E2). And a forest is a stochastic model: comparing two configurations with a single `random_state` and few trees
        > often means comparing the noise of the seed (exercise E1).

        ## Check your understanding

        <details><summary>Do I have to optimize n_estimators with cross-validation?</summary>

No: more trees never worsen the generalization (the error drops and levels off). Choose a sufficient number (usually 200-500) checking that the OOB error is flat, and accept the cost in time and memory. Optimizing it with a grid is almost always a waste of computation.

</details>

<details><summary>When is the OOB error less reliable than cross-validation?</summary>

With duplicate or correlated rows (same customer, repeated measurements, time series): the OOB assumes that an example outside the sample of a tree is independent from those inside, but if a copy of it is in the sample the tree has already 'seen' it and the OOB is optimistic (exercise C1). In those cases you need validation by group or by time.

</details>